# Tradename Cleaning & Validation Pipeline

Clean and validate brand/trade names: remove dosage forms and pack counts, validate via LLM.

## Setup & Configuration

In [ ]:
import glob
import json
import os
import re
import time
from datetime import datetime

import pandas as pd
import requests

# Google Colab setup (uncomment if needed)
# from google.colab import drive
# drive.mount('/content/drive')

BASE_DIR = '/content/drive/MyDrive/DataDoseClean/Tradename Clean'

INPUT_CSV = os.path.join(BASE_DIR, 'DataDoseDataset_FinalV.csv')
OUTPUT_CSV = os.path.join(BASE_DIR, 'dataset_with_validated_tradenames.csv')
OUTPUT_JSON = os.path.join(BASE_DIR, 'tradenames_validated.json')
PROGRESS_FILE = os.path.join(BASE_DIR, 'tradename_pipeline_progress.json')
LOG_FILE = os.path.join(BASE_DIR, 'tradename_pipeline_log.txt')

# API Configuration
GROQ_API_KEYS = []  # Add your Groq API keys
GROQ_MODEL = 'llama-3.1-8b-instant'
GROQ_URL = 'https://api.groq.com/openai/v1/chat/completions'

CONFIDENCE_THRESHOLD = 0.85

print("Configuration loaded.")

## Step 1: Tradename Cleaning - Regex Patterns

In [ ]:
# Regex patterns for removing dosage forms and pack counts

_DOSAGE_FORM_RE = re.compile(
    r"""\b(?:
      f\.?c\.?tab(?:let)?s?\.?   |
      film[- ]coated[- ]tab(?:let)?s?  |
      tab(?:let)?s?              |
      caps?(?:ule)?s?            |
      supp(?:ositorie)?s?        |
      syrup | syr\.?             |
      suspension | susp\.?       |
      solution | sol\.? | soln\.? |
      drops?                     |
      injection | inj\.?         |
      cream | cre\.?             |
      ointment | oint\.?         |
      lotion | lot\.?            |
      powder | pwd\.?            |
      sachets? | sachet          |
      spray | sprays             |
      gel | gels                 |
      patch | patches            |
      disc | discs               |
      lozenge | lozenges         |
      troche | troches           |
      suppository                |
      pessary | pessaries        |
      enema | enemas             |
      implant | implants         |
      device | devices           |
      kit | kits
    )\b""",
    re.IGNORECASE | re.VERBOSE
)

_PACK_VOLUME_RE = re.compile(
    r"""\b(?:
      \d+\s*(?:ml|l|liter|liters|cc|oz|g|gm|kg)\b |
      \d+\s*(?:tabs?|capsules?|caps?|sachets?|vials?|ampoules?|strips?|packs?|boxes?)\b
    )""",
    re.IGNORECASE | re.VERBOSE
)

_LEADING_NUMBER_RE = re.compile(r'^\s*\d+\s+')
_NUMBER_WORD_RE = re.compile(r'\b(?:one|two|three|four|five|six|seven|eight|nine|ten)\b', re.IGNORECASE)

print("Regex patterns compiled.")

## Step 2: Tradename Text Cleaning

In [ ]:
def clean_tradename_text(tradename):
    if not tradename or pd.isna(tradename):
        return None
    
    text = str(tradename).strip()
    if not text:
        return None
    
    # Remove dosage forms
    text = _DOSAGE_FORM_RE.sub('', text)
    
    # Remove pack/volume counts
    text = _PACK_VOLUME_RE.sub('', text)
    
    # Remove leading numbers and number words
    text = _LEADING_NUMBER_RE.sub('', text)
    text = _NUMBER_WORD_RE.sub('', text)
    
    # Standardize whitespace
    text = re.sub(r'\s+', ' ', text).strip()
    
    return text if text else None

# Test samples
test_cases = [
    ('Aspirin 500mg Tablet 20 tabs', 'Aspirin 500mg'),
    ('Amoxicillin Capsule 250mg 30 caps', 'Amoxicillin 250mg'),
    ('1 Paracetamol Syrup 120ml', 'Paracetamol'),
    ('Ibuprofen 200mg Film-Coated Tablet', 'Ibuprofen 200mg'),
]

print("Testing tradename cleaning:")
for raw, _ in test_cases:
    cleaned = clean_tradename_text(raw)
    print(f"  '{raw}' → '{cleaned}'")

## Step 3: LLM Validation

In [ ]:
TRADENAME_VALIDATION_PROMPT = """You are a pharmaceutical product naming specialist.
Validate if the given text is a real, recognized trade/brand name for a drug product.

Task: Return ONLY valid JSON with these fields:
  - is_tradename: true if it's a real pharmaceutical brand name, false otherwise
  - canonical_name: the correct trade name capitalization (or null)
  - confidence: 0.0 to 1.0 confidence score
  - notes: brief reason if rejected

Examples:
  "aspirin" → {"is_tradename": true, "canonical_name": "Aspirin", "confidence": 1.0, "notes": ""}
  "xyz123abc" → {"is_tradename": false, "canonical_name": null, "confidence": 0.95, "notes": "unknown"}
"""

def validate_tradename_with_llm(tradename, api_key=None):
    if not api_key:
        return None
    
    headers = {
        'Authorization': f'Bearer {api_key}',
        'Content-Type': 'application/json',
    }
    
    payload = {
        'model': GROQ_MODEL,
        'messages': [
            {'role': 'system', 'content': TRADENAME_VALIDATION_PROMPT},
            {'role': 'user', 'content': f'Trade name: {tradename}'}
        ],
        'temperature': 0.1,
        'max_tokens': 150,
    }
    
    try:
        response = requests.post(GROQ_URL, json=payload, headers=headers, timeout=10)
        if response.status_code == 200:
            data = response.json()
            content = data['choices'][0]['message']['content']
            return json.loads(content)
    except Exception as e:
        pass
    
    return None

print("LLM validation function ready.")

## Step 4: Full Pipeline

In [ ]:
def run_tradename_pipeline(sample_size=100):
    print(f"Loading data: {INPUT_CSV}")
    df = pd.read_csv(INPUT_CSV)
    
    print(f"Dataset: {len(df):,} rows")
    print(f"Columns: {df.columns.tolist()}")
    
    # Find tradename column
    tradename_col = None
    for col in ['tradename', 'trade_name', 'brand_name', 'brandname']:
        if col in df.columns:
            tradename_col = col
            break
    
    if not tradename_col:
        print("ERROR: Tradename column not found")
        return None
    
    print(f"Using tradename column: '{tradename_col}'")
    
    # Process sample
    results = []
    df_sample = df.head(sample_size).copy()
    
    for idx, row in df_sample.iterrows():
        raw_tradename = row[tradename_col]
        cleaned = clean_tradename_text(raw_tradename)
        
        validation_result = None
        if cleaned and GROQ_API_KEYS:
            validation_result = validate_tradename_with_llm(cleaned, GROQ_API_KEYS[0])
        
        results.append({
            'original_tradename': raw_tradename,
            'tradename_cleaned': cleaned,
            'is_tradename': validation_result.get('is_tradename', False) if validation_result else None,
            'tradename_canonical': validation_result.get('canonical_name') if validation_result else None,
            'confidence': validation_result.get('confidence', 0) if validation_result else None,
        })
        
        if (idx + 1) % 10 == 0:
            print(f"  Processed: {idx + 1} / {sample_size}")
    
    results_df = pd.DataFrame(results)
    results_df.to_csv(OUTPUT_CSV, index=False)
    print(f"\nSaved to: {OUTPUT_CSV}")
    
    print("\nSample results:")
    print(results_df.head(10).to_string())
    
    return results_df

# Uncomment to run
# results = run_tradename_pipeline(sample_size=50)

## Step 5: Filter & Export Validated Tradenames

In [ ]:
def export_validated_tradenames(results_df, confidence_threshold=CONFIDENCE_THRESHOLD):
    if results_df is None:
        return None
    
    df_valid = results_df[
        (results_df['is_tradename'] == True) &
        (results_df['confidence'] >= confidence_threshold)
    ]
    
    print(f"\nFiltering results (confidence >= {confidence_threshold}):")
    print(f"  Total: {len(results_df):,}")
    print(f"  Valid tradenames: {len(df_valid):,}")
    print(f"  Rejected: {len(results_df) - len(df_valid):,}")
    
    # Export JSON
    tradenames_dict = {}
    for _, row in df_valid.iterrows():
        tradenames_dict[row['tradename_cleaned']] = {
            'canonical_name': row['tradename_canonical'],
            'confidence': row['confidence'],
        }
    
    with open(OUTPUT_JSON, 'w', encoding='utf-8') as f:
        json.dump(tradenames_dict, f, indent=2, ensure_ascii=False)
    
    print(f"\nExported:")
    print(f"  CSV: {OUTPUT_CSV}")
    print(f"  JSON: {OUTPUT_JSON}")
    
    return df_valid

## Summary

In [ ]:
print("Tradename Cleaning & Validation Pipeline")
print("=" * 50)
print("\nThis pipeline:")
print("  1. Cleans tradenames: removes dosage forms, pack counts")
print("  2. Validates with LLM: checks if real pharmaceutical brand")
print("  3. Filters by confidence threshold")
print("  4. Exports validated tradenames as CSV and JSON")
print("\nUsage:")
print("  results = run_tradename_pipeline(sample_size=N)")
print("  validated = export_validated_tradenames(results)")